In [2]:
import requests
import pandas as pd
import time

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

# Digamos que você queira montar um dataset só com os desastres mais graves
# "green", "yellow", "orange", "red"
niveis_desejados = ["green", "yellow", "orange", "red"]

lista_sismos_graves = []

print("Buscando terremotos com alertas específicos...")

for alerta in niveis_desejados:
    print(f"Buscando alertas de nível: {alerta.upper()}...")

    parametros = {
        "format": "geojson",
        "starttime": "2008-01-01",  # Pegando um histórico longo
        "endtime": "2026-12-31",
        "alertlevel": alerta        # <--- O parâmetro mágico entra aqui!
        # Note que eu até tirei o 'minmagnitude', pois o alertlevel já atua como um filtro forte
    }

    resposta = requests.get(url, params=parametros)

    if resposta.status_code == 200:
        dados_json = resposta.json()
        sismos = [evento['properties'] for evento in dados_json['features']]
        lista_sismos_graves.extend(sismos)
        print(f"✅ Encontrados {len(sismos)} alertas {alerta}.")
    else:
        print(f"❌ Erro na busca por {alerta}.")

    time.sleep(1) # Pausa amigável para a API

# Transformar em DataFrame
df_graves = pd.DataFrame(lista_sismos_graves)
df_graves['time'] = pd.to_datetime(df_graves['time'], unit='ms')

print(f"\nTotal de terremotos graves baixados: {len(df_graves)}")

Buscando terremotos com alertas específicos...
Buscando alertas de nível: GREEN...
✅ Encontrados 10177 alertas green.
Buscando alertas de nível: YELLOW...
✅ Encontrados 272 alertas yellow.
Buscando alertas de nível: ORANGE...
✅ Encontrados 59 alertas orange.
Buscando alertas de nível: RED...
✅ Encontrados 40 alertas red.

Total de terremotos graves baixados: 10548
